# 1D CNN — Global Cross-Subject Training After Within-Subject 4/1/1 Splits

**NinaPro DB7 | S01–S22 | gestures 13–29 | EMG + ACC | one shared 1D-CNN**

This notebook uses **no LOSO**.

For every subject and gesture:

- **4 complete repetitions → train**
- **1 complete repetition → validation**
- **1 complete repetition → test**

Then the same split type is pooled across all 22 subjects:

- all training repetitions → one global training pool
- all validation repetitions → one global validation pool
- all held-out test repetitions → one global test pool

Terminology: this is **global/cross-subject training with a within-subject repetition
split**. It is not unseen-subject evaluation because every test subject also contributes
different repetitions to the training pool.

In [1]:
import os, gc, time, json, warnings, random, re
from pathlib import Path
from math import gcd

import numpy as np
import pandas as pd

from scipy import io
from scipy.signal import (
    resample_poly,
    butter,
    sosfiltfilt,
    iirnotch,
    filtfilt,
)

import torch
import torch.nn as nn
from torch.utils.data import IterableDataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

print("Imports done.")

Imports done.


In [2]:
def _find_kaggle_input() -> Path:
    base = Path('/kaggle/input')
    if not base.exists():
        return Path('/kaggle/input/ninapro-db7/Dataset')

    def _has_subjects(path):
        try:
            return (
                path.is_dir()
                and any(
                    child.is_dir()
                    and child.name.lower().startswith('subject_')
                    for child in path.iterdir()
                )
            )
        except Exception:
            return False

    def _search(root, depth=0):
        if depth > 5:
            return None
        if _has_subjects(root):
            return root
        try:
            for child in sorted(root.iterdir()):
                if child.is_dir():
                    found = _search(child, depth + 1)
                    if found is not None:
                        return found
        except Exception:
            pass
        return None

    found = _search(base)
    return (
        found
        if found is not None
        else Path('/kaggle/input/ninapro-db7/Dataset')
    )


class Config:
    KAGGLE_INPUT = _find_kaggle_input()
    KAGGLE_WORKING = (
        Path('/kaggle/working')
        if Path('/kaggle').exists()
        else Path.cwd() / '1d_cnn_global_cross_subject_working'
    )

    RUN_TAG = 'v1_1d_cnn_global_cross_subject_r4v1t1_emg_acc'
    REP_CACHE_DIR = KAGGLE_WORKING / f'processed_repetitions_{RUN_TAG}'
    CKPT_DIR = KAGGLE_WORKING / f'ckpts_{RUN_TAG}'
    RESULTS_DIR = KAGGLE_WORKING / f'results_{RUN_TAG}'

    SUBJECTS = list(range(1, 23))
    INTACT_SUBJECTS = list(range(1, 21))
    AMPUTEE_SUBJECTS = [21, 22]

    EXERCISE_IDS = (1, 2)
    GESTURE_MIN = 13
    GESTURE_MAX = 29
    N_CLASSES = GESTURE_MAX - GESTURE_MIN + 1
    REPS_PER_GESTURE = 6
    TRAIN_REPS = 4
    VAL_REPS = 1
    TEST_REPS = 1

    EMG_FS = 2000
    ACC_FS = 148
    TARGET_FS = 2000
    EMG_KEY = 'emg'
    ACC_KEY = 'acc'
    LBL_KEY = 'restimulus'
    N_EMG_CH = 12
    USE_ACC = True

    BANDPASS_LOW_HZ = 20.0
    BANDPASS_HIGH_HZ = 450.0
    FILTER_ORDER = 4
    NOTCH_HZ = 50.0
    NOTCH_Q = 30.0

    WIN_MS = 400
    STEP_MS = 100
    TRIM_MS = 100
    WIN_SAMPLES = int(WIN_MS * TARGET_FS / 1000)
    STEP_SAMPLES = int(STEP_MS * TARGET_FS / 1000)
    TRIM_SAMPLES = int(TRIM_MS * TARGET_FS / 1000)

    DROPOUT = 0.15

    MIN_EPOCHS = 20
    MAX_EPOCHS = 150
    PATIENCE = 15
    MIN_REFIT_EPOCHS = 10
    BATCH_SIZE = 128
    LR = 3e-4
    WEIGHT_DECAY = 1e-4
    GRAD_CLIP = 5.0

    NUM_WORKERS = 0
    KEEP_MODEL = False

    SEED = 42
    DEVICE = torch.device(
        'cuda' if torch.cuda.is_available() else 'cpu'
    )


for directory in [
    Config.REP_CACHE_DIR,
    Config.CKPT_DIR,
    Config.RESULTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

random.seed(Config.SEED)
np.random.seed(Config.SEED)
torch.manual_seed(Config.SEED)
if Config.DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(Config.SEED)

assert Config.USE_ACC is True
assert Config.TRAIN_REPS == 4
assert Config.VAL_REPS == 1
assert Config.TEST_REPS == 1
assert Config.TRAIN_REPS + Config.VAL_REPS + Config.TEST_REPS == 6
assert Config.WIN_MS == 400
assert Config.STEP_MS == 100
assert len(Config.SUBJECTS) == 22

print(f'Device             : {Config.DEVICE}')
print('Evaluation         : GLOBAL CROSS-SUBJECT MODEL, NO LOSO')
print('Within split       : 4 train + 1 validation + 1 test repetition per subject/gesture')
print('Global training    : pool training repetitions from all 22 subjects')
print('Global validation  : pool validation repetitions from all 22 subjects')
print('Global test        : pool held-out test repetitions from all 22 subjects')
print('Model count        : ONE shared 1D-CNN')
print('Input              : raw filtered EMG + repetition-local resampled ACC')
print(f'Window / step      : {Config.WIN_MS} / {Config.STEP_MS} ms')
print(f'Gestures           : {Config.GESTURE_MIN}–{Config.GESTURE_MAX} ({Config.N_CLASSES} classes)')

Device             : cuda
Evaluation         : GLOBAL CROSS-SUBJECT MODEL, NO LOSO
Within split       : 4 train + 1 validation + 1 test repetition per subject/gesture
Global training    : pool training repetitions from all 22 subjects
Global validation  : pool validation repetitions from all 22 subjects
Global test        : pool held-out test repetitions from all 22 subjects
Model count        : ONE shared 1D-CNN
Input              : raw filtered EMG + repetition-local resampled ACC
Window / step      : 400 / 100 ms
Gestures           : 13–29 (17 classes)


## Raw loading and repetition-local preprocessing

In [3]:
class RawEMGACCPreprocessor:
    @staticmethod
    def _find_key(data, candidates):
        for candidate in candidates:
            for key in data:
                if key.lower() == candidate.lower():
                    return key
        return None

    @staticmethod
    def _time_major(array, expected_channels=None, name='signal'):
        array = np.asarray(array)

        if array.ndim == 1:
            array = array[:, None]

        if array.ndim != 2:
            raise RuntimeError(
                f'{name}: expected 2-D array, got {array.shape}.'
            )

        if expected_channels is not None:
            if array.shape[1] == expected_channels:
                return array
            if array.shape[0] == expected_channels:
                return array.T
            raise RuntimeError(
                f'{name}: neither dimension matches '
                f'{expected_channels} channels: {array.shape}.'
            )

        if (
            array.shape[0] < array.shape[1]
            and array.shape[0] <= 128
        ):
            array = array.T

        return array

    def apply(self, mat_path: Path):
        data = io.loadmat(str(mat_path))

        emg_key = self._find_key(data, [Config.EMG_KEY])
        acc_key = self._find_key(data, [Config.ACC_KEY])
        lbl_key = self._find_key(
            data,
            [
                Config.LBL_KEY,
                'stimulus',
                'label',
                'labels',
            ],
        )

        if emg_key is None:
            raise KeyError(f'No EMG key in {mat_path.name}.')
        if acc_key is None:
            raise KeyError(
                f'No ACC key in {mat_path.name}.'
            )
        if lbl_key is None:
            raise KeyError(
                f'No label key in {mat_path.name}.'
            )

        emg = self._time_major(
            data[emg_key],
            expected_channels=Config.N_EMG_CH,
            name=f'{mat_path.name} EMG',
        ).astype(np.float32)

        acc = self._time_major(
            data[acc_key],
            expected_channels=None,
            name=f'{mat_path.name} ACC',
        ).astype(np.float32)

        labels = np.asarray(
            data[lbl_key]
        ).reshape(-1).astype(np.int32)

        n = min(len(emg), len(labels))
        emg = emg[:n]
        labels = labels[:n]

        if len(acc) < 2:
            raise RuntimeError(
                f'{mat_path.name}: ACC has only '
                f'{len(acc)} samples.'
            )

        return emg, acc, labels


class RepetitionEMGFilter:
    def __init__(self):
        nyquist = Config.EMG_FS / 2.0

        self.sos = butter(
            Config.FILTER_ORDER,
            [
                Config.BANDPASS_LOW_HZ / nyquist,
                Config.BANDPASS_HIGH_HZ / nyquist,
            ],
            btype='bandpass',
            output='sos',
        )

        self.b_notch, self.a_notch = iirnotch(
            Config.NOTCH_HZ / nyquist,
            Config.NOTCH_Q,
        )

    def apply(self, repetition_emg):
        repetition_emg = np.asarray(
            repetition_emg,
            dtype=np.float32,
        )

        if repetition_emg.ndim != 2:
            raise ValueError(
                f'Expected (time, channels), got '
                f'{repetition_emg.shape}.'
            )

        if len(repetition_emg) < 64:
            raise RuntimeError(
                f'Repetition unexpectedly short: '
                f'{len(repetition_emg)} samples.'
            )

        filtered = sosfiltfilt(
            self.sos,
            repetition_emg,
            axis=0,
        )
        filtered = filtfilt(
            self.b_notch,
            self.a_notch,
            filtered,
            axis=0,
        )

        return filtered.astype(np.float32)


class RepetitionACCResampler:
    @staticmethod
    def extract_matching_interval(
        file_acc,
        file_emg_length,
        emg_start,
        emg_end,
    ):
        if not (
            0 <= emg_start < emg_end <= file_emg_length
        ):
            raise ValueError(
                f'Invalid EMG interval '
                f'[{emg_start}, {emg_end}).'
            )

        ratio = len(file_acc) / float(file_emg_length)

        acc_start = int(
            np.ceil(emg_start * ratio)
        )
        acc_end = int(
            np.ceil(emg_end * ratio)
        )

        acc_start = max(
            0,
            min(acc_start, len(file_acc) - 1),
        )
        acc_end = max(
            acc_start + 1,
            min(acc_end, len(file_acc)),
        )

        acc_rep = file_acc[
            acc_start:acc_end
        ].copy()

        if len(acc_rep) < 2:
            raise RuntimeError(
                'Mapped ACC repetition is too short.'
            )

        return acc_rep

    @staticmethod
    def resample_to_emg_length(
        acc_rep,
        target_length,
    ):
        source_length = len(acc_rep)
        divisor = gcd(
            source_length,
            target_length,
        )
        up = target_length // divisor
        down = source_length // divisor

        resampled = resample_poly(
            acc_rep,
            up,
            down,
            axis=0,
        ).astype(np.float32)

        if len(resampled) > target_length:
            resampled = resampled[:target_length]
        elif len(resampled) < target_length:
            pad = np.repeat(
                resampled[-1:, :],
                target_length - len(resampled),
                axis=0,
            )
            resampled = np.vstack(
                [resampled, pad]
            )

        if len(resampled) != target_length:
            raise RuntimeError(
                'ACC resampling length mismatch.'
            )

        return resampled.astype(np.float32)


print('Repetition-local EMG filter and ACC resampler defined.')

Repetition-local EMG filter and ACC resampler defined.


## Disk-safe processed repetition cache

Each trimmed EMG+ACC repetition is processed and stored only once.
The 4/1/1 assignment is applied later using repetition metadata, so no overlapping
window cache is created.

In [4]:
class ProcessedRepetitionCache:
    def __init__(self):
        self.preprocessor = RawEMGACCPreprocessor()
        self.emg_filter = RepetitionEMGFilter()
        self.acc_resampler = RepetitionACCResampler()

    def cache_path(self, sid):
        return (
            Config.REP_CACHE_DIR
            / f'S{int(sid):02d}_processed_repetitions.npz'
        )

    def _find_subject_dir(self, sid):
        candidates = [
            Config.KAGGLE_INPUT / f'Subject_{sid}',
            Config.KAGGLE_INPUT / f'subject_{sid}',
            Config.KAGGLE_INPUT / f'S{sid}',
            Config.KAGGLE_INPUT / f's{sid}',
        ]

        for path in candidates:
            if path.is_dir():
                return path

        for path in sorted(
            Config.KAGGLE_INPUT.iterdir()
        ):
            if (
                path.is_dir()
                and path.name.lower().endswith(str(sid))
            ):
                return path

        raise FileNotFoundError(
            f'Cannot find subject {sid} under '
            f'{Config.KAGGLE_INPUT}.'
        )

    @staticmethod
    def _is_exercise_file(path, exercise_id):
        name = path.stem.upper()
        return bool(
            re.search(
                rf'(^|_)E{exercise_id}(_|$)',
                name,
            )
        )

    def _selected_files(self, sid):
        subject_dir = self._find_subject_dir(sid)

        all_mat = (
            sorted(subject_dir.glob('*.mat'))
            or sorted(subject_dir.rglob('*.mat'))
        )

        selected = []

        for exercise_id in Config.EXERCISE_IDS:
            matches = [
                path
                for path in all_mat
                if self._is_exercise_file(
                    path,
                    exercise_id,
                )
            ]

            if not matches:
                raise FileNotFoundError(
                    f'S{sid:02d}: missing E{exercise_id}. '
                    f'Available: '
                    f'{[p.name for p in all_mat[:15]]}'
                )

            selected.extend(matches)

        return selected

    @staticmethod
    def _constant_label_runs(labels):
        if len(labels) == 0:
            return

        boundaries = (
            np.flatnonzero(
                np.diff(labels) != 0
            )
            + 1
        )

        starts = np.r_[0, boundaries]
        ends = np.r_[boundaries, len(labels)]

        for start, end in zip(starts, ends):
            yield (
                int(start),
                int(end),
                int(labels[start]),
            )

    def build_subject(self, sid):
        path = self.cache_path(sid)

        if path.exists():
            with np.load(
                path,
                allow_pickle=False,
            ) as data:
                n_acc_ch = int(
                    data['n_acc_ch']
                )
                metadata = json.loads(
                    str(
                        data[
                            'metadata_json'
                        ].item()
                    )
                )

            print(
                f'  S{sid:02d}: reuse processed '
                f'repetition cache | '
                f'{len(metadata)} reps | '
                f'ACC={n_acc_ch} ch'
            )
            return n_acc_ch, metadata

        runs_by_gesture = {
            gesture: []
            for gesture in range(
                Config.GESTURE_MIN,
                Config.GESTURE_MAX + 1,
            )
        }

        source_parts = []
        acc_counts = set()

        for mat_file in self._selected_files(sid):
            emg, acc, labels = (
                self.preprocessor.apply(mat_file)
            )

            acc_counts.add(
                int(acc.shape[1])
            )

            part_index = len(source_parts)
            source_parts.append({
                'file_name': mat_file.name,
                'emg': emg,
                'acc': acc,
                'labels': labels,
            })

            for (
                run_start,
                run_end,
                gesture,
            ) in self._constant_label_runs(labels):
                if gesture in runs_by_gesture:
                    runs_by_gesture[
                        gesture
                    ].append({
                        'part_index': part_index,
                        'start': run_start,
                        'end': run_end,
                    })

        if len(acc_counts) != 1:
            raise RuntimeError(
                f'S{sid:02d}: inconsistent ACC '
                f'channel counts: {sorted(acc_counts)}'
            )

        n_acc_ch = int(
            next(iter(acc_counts))
        )

        payload = {
            'n_acc_ch': np.array(
                n_acc_ch,
                dtype=np.int64,
            )
        }
        metadata = []

        for gesture in range(
            Config.GESTURE_MIN,
            Config.GESTURE_MAX + 1,
        ):
            runs = runs_by_gesture[gesture]

            if len(runs) != Config.REPS_PER_GESTURE:
                raise RuntimeError(
                    f'S{sid:02d}, gesture {gesture}: '
                    f'expected exactly '
                    f'{Config.REPS_PER_GESTURE} repetitions, '
                    f'found {len(runs)}.'
                )

            for rep_index, record in enumerate(runs):
                part = source_parts[
                    record['part_index']
                ]
                start = record['start']
                end = record['end']

                emg_raw = part['emg'][
                    start:end
                ].copy()

                acc_raw = (
                    self.acc_resampler
                    .extract_matching_interval(
                        file_acc=part['acc'],
                        file_emg_length=len(
                            part['emg']
                        ),
                        emg_start=start,
                        emg_end=end,
                    )
                )

                emg_filtered = (
                    self.emg_filter.apply(
                        emg_raw
                    )
                )

                acc_resampled = (
                    self.acc_resampler
                    .resample_to_emg_length(
                        acc_raw,
                        target_length=len(
                            emg_filtered
                        ),
                    )
                )

                combined = np.concatenate(
                    [
                        emg_filtered,
                        acc_resampled,
                    ],
                    axis=1,
                ).astype(np.float32)

                clean_start = Config.TRIM_SAMPLES
                clean_end = (
                    len(combined)
                    - Config.TRIM_SAMPLES
                )

                if (
                    clean_end - clean_start
                    < Config.WIN_SAMPLES
                ):
                    raise RuntimeError(
                        f'S{sid:02d}, gesture {gesture}, '
                        f'rep {rep_index + 1}: '
                        'too short after trim.'
                    )

                clean = combined[
                    clean_start:clean_end
                ].astype(np.float32)

                key = (
                    f'g{gesture:02d}_'
                    f'r{rep_index + 1:02d}'
                )

                n_windows = (
                    1
                    + (
                        len(clean)
                        - Config.WIN_SAMPLES
                    )
                    // Config.STEP_SAMPLES
                )

                payload[key] = clean
                metadata.append({
                    'key': key,
                    'gesture': int(gesture),
                    'class_id': int(
                        gesture - Config.GESTURE_MIN
                    ),
                    'repetition': int(
                        rep_index + 1
                    ),
                    'n_samples': int(
                        len(clean)
                    ),
                    'n_windows': int(
                        n_windows
                    ),
                    'source_file': (
                        part['file_name']
                    ),
                })

        expected_reps = (
            Config.N_CLASSES
            * Config.REPS_PER_GESTURE
        )

        if len(metadata) != expected_reps:
            raise RuntimeError(
                f'S{sid:02d}: expected '
                f'{expected_reps} processed reps, '
                f'got {len(metadata)}.'
            )

        payload[
            'metadata_json'
        ] = np.array(
            json.dumps(metadata)
        )

        np.savez_compressed(
            path,
            **payload,
        )

        print(
            f'  S{sid:02d}: saved processed '
            f'repetitions | {len(metadata)} reps | '
            f'ACC={n_acc_ch} ch | '
            f'{path.stat().st_size / (1024**2):.1f} MB'
        )

        del source_parts, payload
        gc.collect()

        return n_acc_ch, metadata

    def build_all(self, subjects):
        acc_counts = set()
        all_metadata = {}

        for sid in subjects:
            n_acc_ch, metadata = (
                self.build_subject(sid)
            )
            acc_counts.add(n_acc_ch)
            all_metadata[
                str(sid)
            ] = metadata

        if len(acc_counts) != 1:
            raise RuntimeError(
                'ACC channel count differs '
                f'across subjects: {sorted(acc_counts)}'
            )

        return (
            int(next(iter(acc_counts))),
            all_metadata,
        )

    def load_metadata(self, sid):
        with np.load(
            self.cache_path(sid),
            allow_pickle=False,
        ) as data:
            return json.loads(
                str(
                    data[
                        'metadata_json'
                    ].item()
                )
            )


print('Disk-safe processed-repetition cache defined.')

Disk-safe processed-repetition cache defined.


## Deterministic 4/1/1 repetition assignment

In [5]:
def split_repetition_indices(sid, gesture):
    rng = np.random.default_rng(
        Config.SEED
        + 1009 * int(sid)
        + 9176 * int(gesture)
    )

    perm = rng.permutation(
        Config.REPS_PER_GESTURE
    ).tolist()

    test_idx = sorted(
        perm[:Config.TEST_REPS]
    )
    val_idx = sorted(
        perm[
            Config.TEST_REPS:
            Config.TEST_REPS + Config.VAL_REPS
        ]
    )
    train_idx = sorted(
        perm[
            Config.TEST_REPS + Config.VAL_REPS:
        ]
    )

    assert len(train_idx) == 4
    assert len(val_idx) == 1
    assert len(test_idx) == 1
    assert set(train_idx).isdisjoint(val_idx)
    assert set(train_idx).isdisjoint(test_idx)
    assert set(val_idx).isdisjoint(test_idx)

    return train_idx, val_idx, test_idx


def build_global_assignment(subjects):
    assignment = {}

    for sid in subjects:
        assignment[int(sid)] = {}

        for gesture in range(
            Config.GESTURE_MIN,
            Config.GESTURE_MAX + 1,
        ):
            train_idx, val_idx, test_idx = (
                split_repetition_indices(
                    sid,
                    gesture,
                )
            )

            assignment[int(sid)][int(gesture)] = {
                'train': {
                    int(i + 1)
                    for i in train_idx
                },
                'val': {
                    int(i + 1)
                    for i in val_idx
                },
                'test': {
                    int(i + 1)
                    for i in test_idx
                },
            }

    return assignment


GLOBAL_ASSIGNMENT = build_global_assignment(
    Config.SUBJECTS
)

print('Deterministic subject-wise 4/1/1 assignment defined.')

Deterministic subject-wise 4/1/1 assignment defined.


## Split-aware streaming channel normalizer

In [6]:
class SplitAwareStreamingChannelNormalizer:
    def __init__(self):
        self.mean = None
        self.std = None

    @staticmethod
    def _window_weights(length):
        if length < Config.WIN_SAMPLES:
            raise ValueError(
                'Repetition is shorter than one window.'
            )

        starts = np.arange(
            0,
            length - Config.WIN_SAMPLES + 1,
            Config.STEP_SAMPLES,
            dtype=np.int64,
        )

        diff = np.zeros(
            length + 1,
            dtype=np.int64,
        )

        np.add.at(
            diff,
            starts,
            1,
        )
        np.add.at(
            diff,
            starts + Config.WIN_SAMPLES,
            -1,
        )

        return np.cumsum(
            diff[:-1]
        ).astype(np.float64)

    def fit_from_split(
        self,
        cache_manager,
        subjects,
        assignment,
        split_names,
    ):
        split_names = tuple(
            split_names
        )

        sums = None
        sums_sq = None
        total = 0.0

        for sid in subjects:
            path = (
                cache_manager
                .cache_path(
                    sid
                )
            )

            with np.load(
                path,
                allow_pickle=False,
            ) as data:
                metadata = json.loads(
                    str(
                        data[
                            'metadata_json'
                        ].item()
                    )
                )

                for item in metadata:
                    gesture = int(
                        item[
                            'gesture'
                        ]
                    )
                    repetition = int(
                        item[
                            'repetition'
                        ]
                    )

                    allowed = any(
                        repetition
                        in assignment[
                            int(sid)
                        ][
                            gesture
                        ][
                            split_name
                        ]
                        for split_name
                        in split_names
                    )

                    if not allowed:
                        continue

                    signal = data[
                        item[
                            'key'
                        ]
                    ].astype(
                        np.float64
                    )

                    weights = (
                        self._window_weights(
                            len(
                                signal
                            )
                        )
                    )

                    weighted = (
                        signal
                        * weights[
                            :,
                            None,
                        ]
                    )

                    current_sum = np.sum(
                        weighted,
                        axis=0,
                    )

                    current_sq = np.sum(
                        signal
                        * signal
                        * weights[
                            :,
                            None,
                        ],
                        axis=0,
                    )

                    current_count = float(
                        np.sum(
                            weights
                        )
                    )

                    if sums is None:
                        sums = np.zeros_like(
                            current_sum
                        )
                        sums_sq = np.zeros_like(
                            current_sq
                        )

                    sums += current_sum
                    sums_sq += current_sq
                    total += current_count

        if sums is None or total <= 0:
            raise RuntimeError(
                'No samples available for normalization.'
            )

        mean = sums / total
        variance = (
            sums_sq / total
            - mean * mean
        )
        variance = np.maximum(
            variance,
            1e-12,
        )

        self.mean = mean.astype(
            np.float32
        )
        self.std = np.sqrt(
            variance
        ).astype(
            np.float32
        )

        self.std = np.maximum(
            self.std,
            1e-6,
        )

        return self

    def transform_window(
        self,
        window,
    ):
        if self.mean is None:
            raise RuntimeError(
                'Normalizer has not been fitted.'
            )

        return (
            (
                window
                - self.mean[
                    :,
                    None,
                ]
            )
            / self.std[
                :,
                None,
            ]
        ).astype(
            np.float32
        )


print('Split-aware streaming channel normalizer defined.')

Split-aware streaming channel normalizer defined.


## Streaming windows from selected repetitions only

In [7]:
class GlobalSplitWindowDataset(
    IterableDataset
):
    def __init__(
        self,
        cache_manager,
        subjects,
        assignment,
        split_names,
        normalizer,
        shuffle,
        seed,
    ):
        super().__init__()

        self.cache_manager = (
            cache_manager
        )
        self.subjects = [
            int(sid)
            for sid in subjects
        ]
        self.assignment = (
            assignment
        )
        self.split_names = tuple(
            split_names
        )
        self.normalizer = (
            normalizer
        )
        self.shuffle = bool(
            shuffle
        )
        self.seed = int(
            seed
        )
        self._iteration = 0

        self._length = 0

        for sid in self.subjects:
            for item in (
                cache_manager
                .load_metadata(
                    sid
                )
            ):
                gesture = int(
                    item[
                        'gesture'
                    ]
                )
                repetition = int(
                    item[
                        'repetition'
                    ]
                )

                allowed = any(
                    repetition
                    in assignment[
                        sid
                    ][
                        gesture
                    ][
                        split_name
                    ]
                    for split_name
                    in self.split_names
                )

                if allowed:
                    self._length += int(
                        item[
                            'n_windows'
                        ]
                    )

    def __len__(self):
        return int(
            self._length
        )

    def __iter__(self):
        iteration = (
            self._iteration
        )
        self._iteration += 1

        rng = np.random.default_rng(
            self.seed
            + 104729
            * iteration
        )

        subject_order = (
            self.subjects.copy()
        )

        if self.shuffle:
            rng.shuffle(
                subject_order
            )

        for sid in subject_order:
            path = (
                self.cache_manager
                .cache_path(
                    sid
                )
            )

            with np.load(
                path,
                allow_pickle=False,
            ) as data:
                metadata = json.loads(
                    str(
                        data[
                            'metadata_json'
                        ].item()
                    )
                )

                eligible = []

                for item in metadata:
                    gesture = int(
                        item[
                            'gesture'
                        ]
                    )
                    repetition = int(
                        item[
                            'repetition'
                        ]
                    )

                    allowed = any(
                        repetition
                        in self.assignment[
                            sid
                        ][
                            gesture
                        ][
                            split_name
                        ]
                        for split_name
                        in self.split_names
                    )

                    if allowed:
                        eligible.append(
                            item
                        )

                order = np.arange(
                    len(
                        eligible
                    )
                )

                if self.shuffle:
                    rng.shuffle(
                        order
                    )

                for index in order:
                    item = eligible[
                        int(
                            index
                        )
                    ]

                    signal = data[
                        item[
                            'key'
                        ]
                    ].astype(
                        np.float32
                    )

                    starts = np.arange(
                        0,
                        len(
                            signal
                        )
                        - Config.WIN_SAMPLES
                        + 1,
                        Config.STEP_SAMPLES,
                        dtype=np.int64,
                    )

                    if self.shuffle:
                        rng.shuffle(
                            starts
                        )

                    label = int(
                        item[
                            'class_id'
                        ]
                    )

                    for start in starts:
                        window = signal[
                            int(
                                start
                            ):
                            int(
                                start
                            )
                            + Config.WIN_SAMPLES
                        ].T.copy()

                        window = (
                            self.normalizer
                            .transform_window(
                                window
                            )
                        )

                        yield (
                            torch.from_numpy(
                                window
                            ).float(),
                            torch.tensor(
                                label,
                                dtype=torch.long,
                            ),
                        )


def make_global_loader(
    cache_manager,
    subjects,
    assignment,
    split_names,
    normalizer,
    shuffle,
    seed,
):
    dataset = (
        GlobalSplitWindowDataset(
            cache_manager=(
                cache_manager
            ),
            subjects=(
                subjects
            ),
            assignment=(
                assignment
            ),
            split_names=(
                split_names
            ),
            normalizer=(
                normalizer
            ),
            shuffle=(
                shuffle
            ),
            seed=(
                seed
            ),
        )
    )

    loader = DataLoader(
        dataset,
        batch_size=(
            Config.BATCH_SIZE
        ),
        num_workers=(
            Config.NUM_WORKERS
        ),
        pin_memory=(
            Config.DEVICE.type
            == 'cuda'
        ),
    )

    return (
        loader,
        len(
            dataset
        ),
    )


print('Global split-aware streaming dataset defined.')

Global split-aware streaming dataset defined.


## 1D CNN

In [8]:
class ResidualBlock1D(nn.Module):
    """
    Standard 1D residual block: two conv-bn layers with a skip connection
    added back before the final ReLU. When channels or stride change, the
    skip path goes through a 1x1 conv+BN projection so shapes still match
    for the addition. This lets gradients flow directly through the skip
    path in deep stacks, which plain sequential CNNs don't have.
    """

    def __init__(self, in_ch, out_ch, kernel=5, stride=1, dropout=0.0):
        super().__init__()
        pad = kernel // 2

        self.conv1 = nn.Conv1d(
            in_ch, out_ch, kernel, stride=stride, padding=pad, bias=False
        )
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(
            out_ch, out_ch, kernel, stride=1, padding=pad, bias=False
        )
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.drop = nn.Dropout1d(dropout) if dropout > 0 else nn.Identity()

        needs_projection = (stride != 1) or (in_ch != out_ch)
        if needs_projection:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm1d(out_ch),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))

        out = out + identity
        return self.relu(out)


class Residual1DCNN(nn.Module):
    """
    1D ResNet-style CNN: a conv stem followed by three residual stages
    (64 -> 128 -> 256 channels, kernel 5,5,3), each stage with two
    residual blocks, pooling between stages. Global average pooling +
    the same classifier head used by the other models in this notebook.
    """

    def __init__(self, n_channels, n_classes, dropout=Config.DROPOUT):
        super().__init__()
        self.n_channels = n_channels

        self.stem = nn.Sequential(
            nn.Conv1d(n_channels, 64, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
        )

        self.layer1 = nn.Sequential(
            ResidualBlock1D(64, 64, kernel=5, stride=1, dropout=dropout),
            ResidualBlock1D(64, 64, kernel=5, stride=1, dropout=dropout),
        )
        self.pool1 = nn.MaxPool1d(2)

        self.layer2 = nn.Sequential(
            ResidualBlock1D(64, 128, kernel=5, stride=1, dropout=dropout),
            ResidualBlock1D(128, 128, kernel=5, stride=1, dropout=dropout),
        )
        self.pool2 = nn.MaxPool1d(2)

        self.layer3 = nn.Sequential(
            ResidualBlock1D(128, 256, kernel=3, stride=1, dropout=dropout),
            ResidualBlock1D(256, 256, kernel=3, stride=1, dropout=dropout),
        )

        self.gap = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.pool1(x)
        x = self.layer2(x)
        x = self.pool2(x)
        x = self.layer3(x)
        x = self.gap(x).squeeze(-1)
        return self.head(x)

    def count_params(self):
        return sum(
            parameter.numel()
            for parameter in self.parameters()
            if parameter.requires_grad
        )


print('Residual1DCNN defined.')

Residual1DCNN defined.


## Trainer

In [9]:
class Trainer:
    def __init__(
        self,
        model,
        save_path,
    ):
        self.model = model.to(
            Config.DEVICE
        )
        self.save_path = Path(
            save_path
        )

        self.history = {
            'train_loss': [],
            'val_loss': [],
            'train_acc': [],
            'val_acc': [],
        }

        self.best_epoch = 0
        self.best_val_loss = float(
            'inf'
        )
        self.train_wall = 0.0

    def _run_epoch(
        self,
        loader,
        optimizer=None,
        criterion=None,
    ):
        training = (
            optimizer is not None
        )
        self.model.train(training)

        total_loss = 0.0
        correct = 0
        total = 0

        context = (
            torch.enable_grad()
            if training
            else torch.no_grad()
        )

        with context:
            for X, y in loader:
                X = X.to(
                    Config.DEVICE,
                    non_blocking=True,
                )
                y = y.to(
                    Config.DEVICE,
                    non_blocking=True,
                )

                logits = self.model(X)
                loss = criterion(
                    logits,
                    y,
                )

                if training:
                    optimizer.zero_grad(
                        set_to_none=True
                    )
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(),
                        Config.GRAD_CLIP,
                    )
                    optimizer.step()

                total_loss += (
                    loss.item()
                    * len(y)
                )
                correct += (
                    logits.argmax(1)
                    == y
                ).sum().item()
                total += len(y)

        return (
            total_loss
            / max(total, 1),
            correct
            / max(total, 1),
        )

    def fit(
        self,
        train_loader,
        val_loader,
    ):
        optimizer = Adam(
            self.model.parameters(),
            lr=Config.LR,
            weight_decay=(
                Config.WEIGHT_DECAY
            ),
        )

        criterion = (
            nn.CrossEntropyLoss()
        )

        scheduler = (
            CosineAnnealingLR(
                optimizer,
                T_max=Config.MAX_EPOCHS,
            )
        )

        patience_count = 0
        start_wall = (
            time.perf_counter()
        )

        for epoch in range(
            1,
            Config.MAX_EPOCHS + 1,
        ):
            (
                train_loss,
                train_acc,
            ) = self._run_epoch(
                train_loader,
                optimizer,
                criterion,
            )

            (
                val_loss,
                val_acc,
            ) = self._run_epoch(
                val_loader,
                criterion=criterion,
            )

            scheduler.step()

            self.history[
                'train_loss'
            ].append(train_loss)
            self.history[
                'val_loss'
            ].append(val_loss)
            self.history[
                'train_acc'
            ].append(train_acc)
            self.history[
                'val_acc'
            ].append(val_acc)

            if (
                val_loss
                < self.best_val_loss
            ):
                self.best_val_loss = float(
                    val_loss
                )
                self.best_epoch = int(
                    epoch
                )
                patience_count = 0

                torch.save(
                    self.model.state_dict(),
                    self.save_path,
                )
            else:
                patience_count += 1

            if (
                epoch == 1
                or epoch % 10 == 0
            ):
                print(
                    f'  Epoch {epoch:3d} | '
                    f'train_loss={train_loss:.4f} '
                    f'train_acc={train_acc:.4f} | '
                    f'val_loss={val_loss:.4f} '
                    f'val_acc={val_acc:.4f}'
                )

            if (
                epoch >= Config.MIN_EPOCHS
                and patience_count
                >= Config.PATIENCE
            ):
                print(
                    f'  Early stop at epoch '
                    f'{epoch}'
                )
                break

        self.train_wall = (
            time.perf_counter()
            - start_wall
        )

        print(
            f'  Selection training: '
            f'{self.train_wall:.1f} s | '
            f'best epoch={self.best_epoch} | '
            f'best val loss='
            f'{self.best_val_loss:.4f}'
        )

        return self

    def fit_fixed_epochs(
        self,
        train_loader,
        epochs,
    ):
        optimizer = Adam(
            self.model.parameters(),
            lr=Config.LR,
            weight_decay=(
                Config.WEIGHT_DECAY
            ),
        )

        criterion = (
            nn.CrossEntropyLoss()
        )

        scheduler = (
            CosineAnnealingLR(
                optimizer,
                T_max=max(
                    int(epochs),
                    1,
                ),
            )
        )

        start_wall = (
            time.perf_counter()
        )

        for epoch in range(
            1,
            int(epochs) + 1,
        ):
            loss, acc = (
                self._run_epoch(
                    train_loader,
                    optimizer,
                    criterion,
                )
            )
            scheduler.step()

            if (
                epoch == 1
                or epoch % 10 == 0
                or epoch
                == int(epochs)
            ):
                print(
                    f'  Refit epoch '
                    f'{epoch:3d}/{int(epochs)} | '
                    f'loss={loss:.4f} | '
                    f'acc={acc:.4f}'
                )

        self.train_wall = (
            time.perf_counter()
            - start_wall
        )

        torch.save(
            self.model.state_dict(),
            self.save_path,
        )

        return self


print('Trainer defined.')

Trainer defined.


## Evaluator

In [10]:
class Evaluator:
    def __init__(
        self,
        model_kwargs,
        save_path,
        class_names,
    ):
        self.model_kwargs = (
            model_kwargs
        )
        self.save_path = Path(
            save_path
        )
        self.class_names = (
            class_names
        )

    def evaluate(
        self,
        test_loader,
        split_tag,
    ):
        model = (
            Residual1DCNN(
                **self.model_kwargs
            )
        )

        model.load_state_dict(
            torch.load(
                self.save_path,
                map_location=Config.DEVICE,
            )
        )

        model = model.to(
            Config.DEVICE
        ).eval()

        y_true = []
        y_pred = []
        y_proba = []

        start_wall = (
            time.perf_counter()
        )

        with torch.no_grad():
            for X, y in test_loader:
                X = X.to(
                    Config.DEVICE
                )

                logits = model(X)
                probabilities = (
                    torch.softmax(
                        logits,
                        dim=1,
                    )
                    .cpu()
                    .numpy()
                )

                y_true.append(
                    y.numpy()
                )
                y_pred.append(
                    logits.argmax(1)
                    .cpu()
                    .numpy()
                )
                y_proba.append(
                    probabilities
                )

        test_wall = (
            time.perf_counter()
            - start_wall
        )

        y_true = np.concatenate(
            y_true
        )
        y_pred = np.concatenate(
            y_pred
        )
        y_proba = np.vstack(
            y_proba
        )

        per_class_acc = {}

        for class_id, name in enumerate(
            self.class_names
        ):
            mask = (
                y_true == class_id
            )
            if np.any(mask):
                per_class_acc[
                    name
                ] = float(
                    accuracy_score(
                        y_true[mask],
                        y_pred[mask],
                    )
                )

        try:
            auc = roc_auc_score(
                y_true,
                y_proba,
                multi_class='ovr',
                average='weighted',
                labels=list(
                    range(
                        Config.N_CLASSES
                    )
                ),
            )
        except Exception:
            auc = float('nan')

        metrics = {
            'split': split_tag,
            'accuracy': float(
                accuracy_score(
                    y_true,
                    y_pred,
                )
            ),
            'balanced_accuracy': float(
                balanced_accuracy_score(
                    y_true,
                    y_pred,
                )
            ),
            'precision_w': float(
                precision_score(
                    y_true,
                    y_pred,
                    average='weighted',
                    zero_division=0,
                )
            ),
            'recall_w': float(
                recall_score(
                    y_true,
                    y_pred,
                    average='weighted',
                    zero_division=0,
                )
            ),
            'f1_w': float(
                f1_score(
                    y_true,
                    y_pred,
                    average='weighted',
                    zero_division=0,
                )
            ),
            'roc_auc_w': float(auc),
            'per_class_acc': (
                per_class_acc
            ),
            'test_wall_s': float(
                test_wall
            ),
            'test_samples': int(
                len(y_true)
            ),
        }

        del model
        gc.collect()

        if Config.DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

        return (
            metrics,
            y_true,
            y_proba,
        )


print('Evaluator defined.')

Evaluator defined.


## One global 1D-CNN

In [11]:
class GlobalCrossSubject1DCNNExperiment:
    def __init__(
        self,
        cache_manager,
        n_channels,
        assignment,
        class_names,
    ):
        self.cache_manager = (
            cache_manager
        )
        self.n_channels = int(
            n_channels
        )
        self.assignment = (
            assignment
        )
        self.class_names = (
            class_names
        )
        self.subjects = (
            Config.SUBJECTS.copy()
        )

    def run(self):
        tag = (
            'GLOBAL_1D_CNN_'
            'ALL22_R4_V1_T1'
        )

        print('\n' + '=' * 72)
        print(
            '  ONE GLOBAL 1D-CNN — ALL 22 SUBJECTS'
        )
        print('=' * 72)

        # ----------------------------------------------------------
        # MODEL SELECTION: pooled TRAIN repetitions only
        # ----------------------------------------------------------
        selection_normalizer = (
            SplitAwareStreamingChannelNormalizer()
            .fit_from_split(
                self.cache_manager,
                self.subjects,
                self.assignment,
                split_names=(
                    'train',
                ),
            )
        )

        train_loader, n_train = (
            make_global_loader(
                self.cache_manager,
                self.subjects,
                self.assignment,
                split_names=(
                    'train',
                ),
                normalizer=(
                    selection_normalizer
                ),
                shuffle=True,
                seed=(
                    Config.SEED
                ),
            )
        )

        val_loader, n_val = (
            make_global_loader(
                self.cache_manager,
                self.subjects,
                self.assignment,
                split_names=(
                    'val',
                ),
                normalizer=(
                    selection_normalizer
                ),
                shuffle=False,
                seed=(
                    Config.SEED
                ),
            )
        )

        model_kwargs = {
            'n_channels': (
                self.n_channels
            ),
            'n_classes': (
                Config.N_CLASSES
            ),
        }

        selection_path = (
            Config.CKPT_DIR
            / f'{tag}_selection.pt'
        )

        selection_trainer = (
            Trainer(
                Residual1DCNN(
                    **model_kwargs
                ),
                selection_path,
            )
        )

        selection_trainer.fit(
            train_loader,
            val_loader,
        )

        # ----------------------------------------------------------
        # FINAL REFIT: pooled TRAIN + VALIDATION only
        # ----------------------------------------------------------
        refit_epochs = max(
            int(
                selection_trainer
                .best_epoch
            ),
            Config.MIN_REFIT_EPOCHS,
        )

        final_normalizer = (
            SplitAwareStreamingChannelNormalizer()
            .fit_from_split(
                self.cache_manager,
                self.subjects,
                self.assignment,
                split_names=(
                    'train',
                    'val',
                ),
            )
        )

        refit_loader, n_refit = (
            make_global_loader(
                self.cache_manager,
                self.subjects,
                self.assignment,
                split_names=(
                    'train',
                    'val',
                ),
                normalizer=(
                    final_normalizer
                ),
                shuffle=True,
                seed=(
                    Config.SEED
                    + 7919
                ),
            )
        )

        test_loader, n_test = (
            make_global_loader(
                self.cache_manager,
                self.subjects,
                self.assignment,
                split_names=(
                    'test',
                ),
                normalizer=(
                    final_normalizer
                ),
                shuffle=False,
                seed=(
                    Config.SEED
                ),
            )
        )

        final_path = (
            Config.CKPT_DIR
            / f'{tag}_refit.pt'
        )

        refit_trainer = (
            Trainer(
                Residual1DCNN(
                    **model_kwargs
                ),
                final_path,
            )
        )

        refit_trainer.fit_fixed_epochs(
            refit_loader,
            refit_epochs,
        )

        evaluator = Evaluator(
            model_kwargs,
            final_path,
            self.class_names,
        )

        (
            metrics,
            y_true,
            y_proba,
        ) = evaluator.evaluate(
            test_loader,
            tag,
        )

        metrics.update({
            'n_subjects': 22,
            'train_windows': int(
                n_train
            ),
            'validation_windows': int(
                n_val
            ),
            'refit_windows': int(
                n_refit
            ),
            'test_windows': int(
                n_test
            ),
            'selection_best_epoch': int(
                selection_trainer
                .best_epoch
            ),
            'refit_epochs': int(
                refit_epochs
            ),
            'n_params': int(
                Residual1DCNN(
                    **model_kwargs
                ).count_params()
            ),
            'same_subjects_across_splits': True,
            'unseen_subject_evaluation': False,
            'use_acc': True,
            'window_ms': int(
                Config.WIN_MS
            ),
            'step_ms': int(
                Config.STEP_MS
            ),
            'evaluation_scope': (
                'global_cross_subject_1d_cnn_'
                'within_subject_r4v1t1_'
                'emg_acc_leakage_safe'
            ),
        })

        print(
            f'\nGLOBAL 1D-CNN TEST | '
            f'Acc={metrics["accuracy"]:.4f} | '
            f'BalAcc={metrics["balanced_accuracy"]:.4f} | '
            f'F1={metrics["f1_w"]:.4f} | '
            f'best/refit='
            f'{selection_trainer.best_epoch}/'
            f'{refit_epochs}'
        )

        if not Config.KEEP_MODEL:
            for path in (
                selection_path,
                final_path,
            ):
                try:
                    if path.exists():
                        path.unlink()
                except Exception:
                    pass

        del (
            train_loader,
            val_loader,
            refit_loader,
            test_loader,
            selection_trainer,
            refit_trainer,
            selection_normalizer,
            final_normalizer,
        )

        gc.collect()

        if Config.DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

        return (
            metrics,
            y_true,
            y_proba,
        )


print('One-global-1D-CNN experiment defined.')

One-global-1D-CNN experiment defined.


## Main pipeline

In [12]:
def main_global_1d_cnn():
    print('\n' + '=' * 72)
    print(
        '  GLOBAL 1D-CNN AFTER WITHIN-SUBJECT 4/1/1 SPLITS'
    )
    print('=' * 72)

    cache_manager = (
        ProcessedRepetitionCache()
    )

    n_acc_ch, all_metadata = (
        cache_manager.build_all(
            Config.SUBJECTS
        )
    )

    n_channels = (
        Config.N_EMG_CH
        + n_acc_ch
    )

    print(
        f'\nInput channels      : '
        f'{n_channels} '
        f'({Config.N_EMG_CH} EMG + '
        f'{n_acc_ch} ACC)'
    )

    serializable_assignment = {}

    for sid, by_gesture in (
        GLOBAL_ASSIGNMENT.items()
    ):
        serializable_assignment[
            str(sid)
        ] = {}

        for gesture, splits in (
            by_gesture.items()
        ):
            serializable_assignment[
                str(sid)
            ][
                str(gesture)
            ] = {
                split_name: sorted(
                    list(
                        reps
                    )
                )
                for (
                    split_name,
                    reps,
                ) in splits.items()
            }

    with open(
        Config.RESULTS_DIR
        / 'global_1d_cnn_repetition_assignments.json',
        'w',
    ) as file:
        json.dump(
            serializable_assignment,
            file,
            indent=2,
        )

    class_names = [
        f'G{gesture:02d}'
        for gesture in range(
            Config.GESTURE_MIN,
            Config.GESTURE_MAX + 1,
        )
    ]

    experiment = (
        GlobalCrossSubject1DCNNExperiment(
            cache_manager=(
                cache_manager
            ),
            n_channels=(
                n_channels
            ),
            assignment=(
                GLOBAL_ASSIGNMENT
            ),
            class_names=(
                class_names
            ),
        )
    )

    (
        metrics,
        y_true,
        y_proba,
    ) = experiment.run()

    serializable = {}

    for key, value in (
        metrics.items()
    ):
        if isinstance(
            value,
            (
                np.integer,
                np.floating,
            ),
        ):
            serializable[
                key
            ] = value.item()
        else:
            serializable[
                key
            ] = value

    with open(
        Config.RESULTS_DIR
        / 'global_1d_cnn_aggregate_results.json',
        'w',
    ) as file:
        json.dump(
            serializable,
            file,
            indent=2,
        )

    print('\n' + '=' * 72)
    print(
        '  FINAL GLOBAL CROSS-SUBJECT 1D-CNN SUMMARY'
    )
    print('=' * 72)

    print(
        f'Pooled global test accuracy : '
        f'{metrics["accuracy"]:.4f}'
    )
    print(
        f'Pooled balanced accuracy    : '
        f'{metrics["balanced_accuracy"]:.4f}'
    )
    print(
        f'Pooled weighted F1          : '
        f'{metrics["f1_w"]:.4f}'
    )
    print(
        f'Test windows                : '
        f'{metrics["test_windows"]:,}'
    )

    print('\nProtocol interpretation:')
    print(
        '  One shared/global 1D-CNN is trained on pooled '
        'training repetitions from all 22 subjects.'
    )
    print(
        '  Held-out test repetitions are unseen repetitions, '
        'but their subjects also occur in training.'
    )
    print(
        '  This is global cross-subject training with '
        'a within-subject repetition split; it is not LOSO.'
    )

    print('\nLeakage / disk controls:')
    print('  ✓ every subject/gesture is split 4/1/1 by complete repetitions')
    print('  ✓ train/validation/test repetition identities are disjoint')
    print('  ✓ zero-phase EMG filtering is repetition-local')
    print('  ✓ ACC extraction/resampling is repetition-local and same-source-file')
    print('  ✓ processed repetitions are cached once; overlapping windows are never cached')
    print('  ✓ selection normalizer uses pooled training repetitions only')
    print('  ✓ validation repetitions select the epoch only')
    print('  ✓ fresh final normalizer uses pooled train+validation only')
    print('  ✓ fresh final 1D-CNN trains on pooled train+validation only')
    print('  ✓ pooled test repetitions never affect learned normalization')
    print('  ✓ one shared 1D-CNN for all 22 subjects')
    print('  ✓ no LOSO and no random window-level train/test split')

    return (
        metrics,
        y_true,
        y_proba,
        serializable_assignment,
    )


if __name__ == '__main__':
    (
        global_metrics,
        all_y_true,
        all_y_proba,
        repetition_assignments,
    ) = main_global_1d_cnn()


  GLOBAL 1D-CNN AFTER WITHIN-SUBJECT 4/1/1 SPLITS
  S01: saved processed repetitions | 102 reps | ACC=36 ch | 134.7 MB
  S02: saved processed repetitions | 102 reps | ACC=36 ch | 157.9 MB
  S03: saved processed repetitions | 102 reps | ACC=36 ch | 127.1 MB
  S04: saved processed repetitions | 102 reps | ACC=36 ch | 166.8 MB
  S05: saved processed repetitions | 102 reps | ACC=36 ch | 116.5 MB
  S06: saved processed repetitions | 102 reps | ACC=36 ch | 127.2 MB
  S07: saved processed repetitions | 102 reps | ACC=36 ch | 100.2 MB
  S08: saved processed repetitions | 102 reps | ACC=36 ch | 118.1 MB
  S09: saved processed repetitions | 102 reps | ACC=36 ch | 128.2 MB
  S10: saved processed repetitions | 102 reps | ACC=36 ch | 115.1 MB
  S11: saved processed repetitions | 102 reps | ACC=36 ch | 169.2 MB
  S12: saved processed repetitions | 102 reps | ACC=36 ch | 120.0 MB
  S13: saved processed repetitions | 102 reps | ACC=36 ch | 133.0 MB
  S14: saved processed repetitions | 102 reps | ACC=